In [1]:
from datetime import datetime, timedelta
import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.layers import Dense, Layer

# Fix de semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


# Definición de la capa de atención personalizada
class AtencionTemporal(Layer):

  def __init__(self, **kwargs):
    super().__init__(**kwargs)
    self.score_dense = Dense(1)

  def call(self, secuencia):
    scores = self.score_dense(secuencia)
    pesos = tf.nn.softmax(scores, axis=1)
    # reduce_sum colapsa la dimensión temporal (eje 1) devolviendo un tensor 2D
    return tf.reduce_sum(secuencia * pesos, axis=1)


print('✓ Librerías y clase AtencionTemporal registradas correctamente.')

✓ Librerías y clase AtencionTemporal registradas correctamente.


In [3]:
nombres_archivos = [
    'mejor_red1_long.keras',  # Índice 0: TCN Sinóptico (6h, 12h)
    'mejor_red1_short.keras',  # Índice 1: CNN-LSTM Corto (15m, 1h, 3h)
    'mejor_red2.keras',  # Índice 2: Regresión de Cuantiles Monto
    'mejor_red3_temperatura.keras',  # Índice 3: Regresión Temperatura
]

modelos = []

for nombre in nombres_archivos:
  try:
    m = tf.keras.models.load_model(
        nombre,
        custom_objects={'AtencionTemporal': AtencionTemporal},
        compile=False,
  )
    modelos.append(m)
    print(f'✓ Cargado con éxito: {nombre}')
  except Exception as e:
    print(f'✗ Error al cargar {nombre}: {e}')

✓ Cargado con éxito: mejor_red1_long.keras
✓ Cargado con éxito: mejor_red1_short.keras
✓ Cargado con éxito: mejor_red2.keras
✓ Cargado con éxito: mejor_red3_temperatura.keras


In [4]:
# 1. Parámetros de la consulta
latitude, longitude = 19.09, -98.26
tz = 'America/Mexico_City'
variables_api = [
    'precipitation',
    'rain',
    'wind_speed_10m',
    'dew_point_2m',
    'relative_humidity_2m',
    'cloud_cover',
    'temperature_2m',
]
col_mapping = {
    'precipitation': 'precipitation (mm)',
    'rain': 'rain (mm)',
    'wind_speed_10m': 'wind_speed_10m (km/h)',
    'dew_point_2m': 'dew_point_2m (°C)',
    'relative_humidity_2m': 'relative_humidity_2m (%)',
    'cloud_cover': 'cloud_cover (%)',
    'temperature_2m': 'temperature_2m (°C)',
}

today = datetime.now()
start_date = today - timedelta(days=3)

print('Descargando datos meteorológicos recientes...')
params = {
    'latitude': latitude,
    'longitude': longitude,
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': today.strftime('%Y-%m-%d'),
    'hourly': ','.join(variables_api),
    'timezone': tz,
}
res = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
res.raise_for_status()
raw_data = res.json()['hourly']

df_hourly = pd.DataFrame({'time': pd.to_datetime(raw_data['time'])})
for var in variables_api:
  df_hourly[col_mapping.get(var, var)] = raw_data[var]

df_hourly = df_hourly.set_index('time').sort_index()
hora_limite = pd.Timestamp.now(tz=tz).tz_localize(None).ceil('h')
df_hourly = df_hourly[df_hourly.index <= hora_limite]

# 2. Resampling a 15 min + Integrales Newton-Cotes (1h)
df_15min = df_hourly.resample('15min').interpolate(method='linear')

COLS_EPISODICAS = ['rain (mm)', 'precipitation (mm)']
COLS_CONTINUAS = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'dew_point_2m (°C)',
    'wind_speed_10m (km/h)',
    'cloud_cover (%)',
]
COLS_BASE = COLS_EPISODICAS + COLS_CONTINUAS
SUFIJO_INTEGRAL = '_integral_1hora'

for col in COLS_BASE:
  df_15min[col + SUFIJO_INTEGRAL] = (
      df_15min[col].rolling(4, min_periods=1).mean()
  )

# 3. Feature Engineering de Derivadas
df_15min['dew_point_depression'] = (
    df_15min['dew_point_2m (°C)'] - df_15min['temperature_2m (°C)']
)
df_15min['temperature_2m (°C)_delta_1h'] = df_15min[
    'temperature_2m (°C)'
].diff(4)
df_15min['temperature_2m (°C)_delta_3h'] = df_15min[
    'temperature_2m (°C)'
].diff(12)
df_15min['relative_humidity_2m (%)_delta_1h'] = df_15min[
    'relative_humidity_2m (%)'
].diff(4)
df_15min['relative_humidity_2m (%)_delta_3h'] = df_15min[
    'relative_humidity_2m (%)'
].diff(12)
df_15min['cloud_cover (%)_delta_1h'] = df_15min['cloud_cover (%)'].diff(4)
df_15min['cloud_cover (%)_delta_3h'] = df_15min['cloud_cover (%)'].diff(12)

for ventana_h, pasos in [(1, 4), (3, 12), (6, 24), (24, 96)]:
  df_15min[f'rain_sum_{ventana_h}h'] = (
      df_15min['rain (mm)'].rolling(pasos, min_periods=1).sum()
  )

df_15min['humidity_mean_3h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(12, min_periods=1).mean()
)
df_15min['humidity_std_3h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(12, min_periods=1).std()
)
df_15min['humidity_mean_24h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(96, min_periods=1).mean()
)

esta_lloviendo = df_15min['rain (mm)'] > 0.1
marca_tiempo = df_15min.index.to_series()
ultima_marca_lluvia = marca_tiempo.where(esta_lloviendo).ffill()
horas_desde = (
    marca_tiempo - ultima_marca_lluvia
).dt.total_seconds() / 3600.0
df_15min['horas_desde_ultima_lluvia'] = horas_desde.fillna(999.0).clip(
    upper=999.0
)

# Cíclicas
df_15min['hour_sin'] = np.sin(2 * np.pi * df_15min.index.hour / 24)
df_15min['hour_cos'] = np.cos(2 * np.pi * df_15min.index.hour / 24)
df_15min['day_year_sin'] = np.sin(2 * np.pi * df_15min.index.dayofyear / 365.24)
df_15min['day_year_cos'] = np.cos(2 * np.pi * df_15min.index.dayofyear / 365.24)

df_15min_proc = df_15min.bfill().fillna(0)

# Lista final de las 33 características
COLS_INTEGRAL = [c + SUFIJO_INTEGRAL for c in COLS_BASE]
COLS_DERIVADAS = [
    'dew_point_depression',
    'temperature_2m (°C)_delta_1h',
    'temperature_2m (°C)_delta_3h',
    'relative_humidity_2m (%)_delta_1h',
    'relative_humidity_2m (%)_delta_3h',
    'cloud_cover (%)_delta_1h',
    'cloud_cover (%)_delta_3h',
    'rain_sum_1h',
    'rain_sum_3h',
    'rain_sum_6h',
    'rain_sum_24h',
    'humidity_mean_3h',
    'humidity_std_3h',
    'humidity_mean_24h',
    'horas_desde_ultima_lluvia',
]
COLS_CICLICAS = ['hour_sin', 'hour_cos', 'day_year_sin', 'day_year_cos']

FEATURES = COLS_BASE + COLS_INTEGRAL + COLS_DERIVADAS + COLS_CICLICAS
print(f'✓ Dataset procesado con éxito. Total de variables: {len(FEATURES)}')

Descargando datos meteorológicos recientes...
✓ Dataset procesado con éxito. Total de variables: 33


In [5]:
# 1. Parámetros de la consulta
latitude, longitude = 19.09, -98.26
tz = 'America/Mexico_City'
variables_api = [
    'precipitation',
    'rain',
    'wind_speed_10m',
    'dew_point_2m',
    'relative_humidity_2m',
    'cloud_cover',
    'temperature_2m',
]
col_mapping = {
    'precipitation': 'precipitation (mm)',
    'rain': 'rain (mm)',
    'wind_speed_10m': 'wind_speed_10m (km/h)',
    'dew_point_2m': 'dew_point_2m (°C)',
    'relative_humidity_2m': 'relative_humidity_2m (%)',
    'cloud_cover': 'cloud_cover (%)',
    'temperature_2m': 'temperature_2m (°C)',
}

today = datetime.now()
start_date = today - timedelta(days=3)

print('Descargando datos meteorológicos recientes...')
params = {
    'latitude': latitude,
    'longitude': longitude,
    'start_date': start_date.strftime('%Y-%m-%d'),
    'end_date': today.strftime('%Y-%m-%d'),
    'hourly': ','.join(variables_api),
    'timezone': tz,
}
res = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
res.raise_for_status()
raw_data = res.json()['hourly']

df_hourly = pd.DataFrame({'time': pd.to_datetime(raw_data['time'])})
for var in variables_api:
  df_hourly[col_mapping.get(var, var)] = raw_data[var]

df_hourly = df_hourly.set_index('time').sort_index()
hora_limite = pd.Timestamp.now(tz=tz).tz_localize(None).ceil('h')
df_hourly = df_hourly[df_hourly.index <= hora_limite]

# 2. Resampling a 15 min + Integrales Newton-Cotes (1h)
df_15min = df_hourly.resample('15min').interpolate(method='linear')

COLS_EPISODICAS = ['rain (mm)', 'precipitation (mm)']
COLS_CONTINUAS = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'dew_point_2m (°C)',
    'wind_speed_10m (km/h)',
    'cloud_cover (%)',
]
COLS_BASE = COLS_EPISODICAS + COLS_CONTINUAS
SUFIJO_INTEGRAL = '_integral_1hora'

for col in COLS_BASE:
  df_15min[col + SUFIJO_INTEGRAL] = (
      df_15min[col].rolling(4, min_periods=1).mean()
  )

# 3. Feature Engineering de Derivadas
df_15min['dew_point_depression'] = (
    df_15min['dew_point_2m (°C)'] - df_15min['temperature_2m (°C)']
)
df_15min['temperature_2m (°C)_delta_1h'] = df_15min[
    'temperature_2m (°C)'
].diff(4)
df_15min['temperature_2m (°C)_delta_3h'] = df_15min[
    'temperature_2m (°C)'
].diff(12)
df_15min['relative_humidity_2m (%)_delta_1h'] = df_15min[
    'relative_humidity_2m (%)'
].diff(4)
df_15min['relative_humidity_2m (%)_delta_3h'] = df_15min[
    'relative_humidity_2m (%)'
].diff(12)
df_15min['cloud_cover (%)_delta_1h'] = df_15min['cloud_cover (%)'].diff(4)
df_15min['cloud_cover (%)_delta_3h'] = df_15min['cloud_cover (%)'].diff(12)

for ventana_h, pasos in [(1, 4), (3, 12), (6, 24), (24, 96)]:
  df_15min[f'rain_sum_{ventana_h}h'] = (
      df_15min['rain (mm)'].rolling(pasos, min_periods=1).sum()
  )

df_15min['humidity_mean_3h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(12, min_periods=1).mean()
)
df_15min['humidity_std_3h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(12, min_periods=1).std()
)
df_15min['humidity_mean_24h'] = (
    df_15min['relative_humidity_2m (%)'].rolling(96, min_periods=1).mean()
)

esta_lloviendo = df_15min['rain (mm)'] > 0.1
marca_tiempo = df_15min.index.to_series()
ultima_marca_lluvia = marca_tiempo.where(esta_lloviendo).ffill()
horas_desde = (
    marca_tiempo - ultima_marca_lluvia
).dt.total_seconds() / 3600.0
df_15min['horas_desde_ultima_lluvia'] = horas_desde.fillna(999.0).clip(
    upper=999.0
)

# Cíclicas
df_15min['hour_sin'] = np.sin(2 * np.pi * df_15min.index.hour / 24)
df_15min['hour_cos'] = np.cos(2 * np.pi * df_15min.index.hour / 24)
df_15min['day_year_sin'] = np.sin(2 * np.pi * df_15min.index.dayofyear / 365.24)
df_15min['day_year_cos'] = np.cos(2 * np.pi * df_15min.index.dayofyear / 365.24)

df_15min_proc = df_15min.bfill().fillna(0)

# Lista final de las 33 características
COLS_INTEGRAL = [c + SUFIJO_INTEGRAL for c in COLS_BASE]
COLS_DERIVADAS = [
    'dew_point_depression',
    'temperature_2m (°C)_delta_1h',
    'temperature_2m (°C)_delta_3h',
    'relative_humidity_2m (%)_delta_1h',
    'relative_humidity_2m (%)_delta_3h',
    'cloud_cover (%)_delta_1h',
    'cloud_cover (%)_delta_3h',
    'rain_sum_1h',
    'rain_sum_3h',
    'rain_sum_6h',
    'rain_sum_24h',
    'humidity_mean_3h',
    'humidity_std_3h',
    'humidity_mean_24h',
    'horas_desde_ultima_lluvia',
]
COLS_CICLICAS = ['hour_sin', 'hour_cos', 'day_year_sin', 'day_year_cos']

FEATURES = COLS_BASE + COLS_INTEGRAL + COLS_DERIVADAS + COLS_CICLICAS
print(f'✓ Dataset procesado con éxito. Total de variables: {len(FEATURES)}')

Descargando datos meteorológicos recientes...
✓ Dataset procesado con éxito. Total de variables: 33


In [6]:
# 1. Escalado de datos
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df_15min_proc[FEATURES].values)

agg_dict = {c: 'mean' for c in FEATURES}
for c in COLS_EPISODICAS:
  agg_dict[c] = 'sum'
df_horario = df_15min_proc[FEATURES].resample('1h').agg(agg_dict).ffill()

scaler_h = MinMaxScaler(feature_range=(0, 1))
df_horario_scaled = scaler_h.fit_transform(df_horario.values)

# 2. Generación de Tensores de Entrada
Xr_input = data_scaled[-16:, :][np.newaxis, ...].astype('float32')  # (1, 16, 33)
Xt_input = data_scaled[-48:, :][np.newaxis, ...].astype('float32')  # (1, 48, 33)
Xs_input = df_horario_scaled[-48:, :][np.newaxis, ...].astype(
    'float32'
)  # (1, 48, 33)

# 3. Predicciones
prob_short = modelos[1].predict(Xr_input, verbose=0)
prob_short_list = [p.reshape(1, -1) for p in prob_short]
prob_short_arr = np.concatenate(prob_short_list, axis=1)  # Forma (1, 3)

prob_long = modelos[0].predict(Xs_input, verbose=0)
amt_pred = modelos[2].predict(
    [Xr_input, Xt_input, prob_short_arr], verbose=0
)
temp_scaled = modelos[3].predict(Xt_input, verbose=0)[0]

# Desescalar Temperatura
IDX_TEMP = FEATURES.index('temperature_2m (°C)')
dummy = np.zeros((len(temp_scaled), len(FEATURES)), dtype='float32')
dummy[:, IDX_TEMP] = temp_scaled
temp_c = scaler.inverse_transform(dummy)[:, IDX_TEMP]

# 4. Despliegue de Resultados
nombres_salida_red2 = list(modelos[2].output_names)
idx_por_nombre = {n: i for i, n in enumerate(nombres_salida_red2)}

HORIZONTES_CORTOS = ['15min', '1h', '3h']
HORIZONTES_LARGOS = ['6h', '12h']

print('\n================ PRONÓSTICO METEOROLÓGICO ================')
for j, h in enumerate(HORIZONTES_CORTOS):
  prob_h = float(prob_short[j][0, 0])
  p50 = np.expm1(
      float(
          np.clip(
              amt_pred[idx_por_nombre[f'rain_amount_{h}_p50']][0, 0], 0, None
          )
      )
  )
  p10 = np.expm1(
      float(
          np.clip(
              amt_pred[idx_por_nombre[f'rain_amount_{h}_p10']][0, 0], 0, None
          )
      )
  )
  p90 = np.expm1(
      float(
          np.clip(
              amt_pred[idx_por_nombre[f'rain_amount_{h}_p90']][0, 0], 0, None
          )
      )
  )

  print(f'\n▶ Horizonte {h}:')
  print(f'  • Probabilidad de lluvia: {prob_h * 100:.1f}%')
  print(
      f'  • Acumulado (P50): {p50:.2f} mm [Rango P10-P90: {p10:.2f} - {p90:.2f}'
      ' mm]'
  )
  print(f'  • Temperatura: {temp_c[j]:.1f} °C')

for j, h in enumerate(HORIZONTES_LARGOS):
  prob_h = float(prob_long[j][0, 0])
  print(f'\n▶ Horizonte {h} (Visión Sinóptica):')
  print(f'  • Probabilidad de lluvia: {prob_h * 100:.1f}%')
  print(f'  • Temperatura: {temp_c[j + 3]:.1f} °C')


================ PRONÓSTICO METEOROLÓGICO ================

▶ Horizonte 15min:
  • Probabilidad de lluvia: 0.0%
  • Acumulado (P50): 0.00 mm [Rango P10-P90: 0.00 - 0.00 mm]
  • Temperatura: 22.4 °C

▶ Horizonte 1h:
  • Probabilidad de lluvia: 0.2%
  • Acumulado (P50): 0.01 mm [Rango P10-P90: 0.00 - 0.01 mm]
  • Temperatura: 22.2 °C

▶ Horizonte 3h:
  • Probabilidad de lluvia: 0.5%
  • Acumulado (P50): 0.03 mm [Rango P10-P90: 0.00 - 0.09 mm]
  • Temperatura: 21.1 °C

▶ Horizonte 6h (Visión Sinóptica):
  • Probabilidad de lluvia: 0.6%
  • Temperatura: 19.1 °C

▶ Horizonte 12h (Visión Sinóptica):
  • Probabilidad de lluvia: 0.8%
  • Temperatura: 15.3 °C
